# Question 1
## Weekly Assessment -Agentic AI
## Description
## Capstone Assessment (Duration: 2 Hours)
## 🔷 Scenario 1: AI Customer Support Multi-Agent System
- 🎯 Objective:
Build a multi-agent customer support system that can classify, delegate, and resolve user queries.

- 💼 Problem Statement
A company wants to automate its support system using AI agents.

### The system should:

- Understand customer queries
- Route them to the correct department
- Provide responses
- 👥 Agents to Build
- Classifier Agent
- Identifies query type (Billing / Technical / General)
- Billing Agent
- Handles payment/refund queries
- Technical Support Agent
- Handles app/software issues
- Response Agent
- Combines outputs into a final response
### ⚙️ Tasks to Implement
Multi-agent architecture (at least 3 agents mandatory)
Task delegation logic
Basic agent communication (message passing / function calls)
### 💡 Sample Inputs
“I was charged twice for my subscription”
“The app crashes when I open it”
“What are your pricing plans?”
### 🧪 Expected Output
Correct classification of query
Delegation to appropriate agent
Final structured response
### 🧰 Tools (Choose Any)
Python (basic functions or LLM APIs)
LangGraph (workflow)
CrewAI (role-based agents)
AutoGen (agent conversation)
Microsoft Copilot Studio (optional low-code)
### 📦 Deliverables
Working code / flow
Architecture diagram
2–3 test case outputs

In [9]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('GROQ_API_KEY'), base_url="https://api.groq.com/openai/v1")

# ============================================
# UNIVERSAL LLM AGENT FUNCTION
# ============================================
def call_llm_agent(persona_name, system_instruction, user_query):
    print(f" [{persona_name}] is processing...")
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_query}
        ],
        temperature=0.2 # Lower temperature = more consistent, less "chatty"
    )
    return response.choices[0].message.content

# ============================================
# DECISIVE AGENT PERSONAS (No more asking questions!)
# ============================================

def billing_agent(query):
    instruction = """You are an Automated Billing Resolver. 
    1. Acknowledge the specific amount and date provided.
    2. Confirm that the refund has been INITIATED in the system.
    3. State that no further action is needed from the user.
    4. DO NOT ask any follow-up questions."""
    return call_llm_agent("Billing Agent", instruction, query)

def technical_agent(query):
    instruction = """You are an Auto-Diagnostic Technical Agent. 
    1. Analyze the specific crash or error mentioned.
    2. Provide ONE immediate technical fix (e.g., clearing cache or updating).
    3. Confirm a diagnostic ticket has been opened.
    4. DO NOT ask the user for more details."""
    return call_llm_agent("Tech Agent", instruction, query)

def general_agent(query):
    instruction = """You are a Direct Info Agent. 
    1. Provide the specific answer requested (pricing, hours, etc.).
    2. If the user is vague, provide the most popular 'Standard' info.
    3. DO NOT ask 'How else can I help?'"""
    return call_llm_agent("General Agent", instruction, query)

# ============================================
# CLASSIFIER & RESPONSE FORMATTER
# ============================================
def classifier_agent(user_query):
    prompt = "Classify this query into: BILLING, TECHNICAL, or GENERAL. Return ONLY the category name."
    return call_llm_agent("Classifier", prompt, user_query).strip().upper()

def response_agent(category, final_content, original_query):
    return f"""
--------------------------------------------------
 AGENTIC RESOLUTION SUMMARY
--------------------------------------------------
DEPT    : {category}
INPUT   : {original_query}
--------------------------------------------------
ACTION TAKEN:
{final_content}
--------------------------------------------------
STATUS  : CLOSED / RESOLVED
--------------------------------------------------
"""

# ============================================
# ORCHESTRATOR
# ============================================
def run_agentic_workflow(user_input):
    category = classifier_agent(user_input)
    
    if "BILLING" in category:
        worker_output = billing_agent(user_input)
    elif "TECHNICAL" in category:
        worker_output = technical_agent(user_input)
    else:
        category = "GENERAL"
        worker_output = general_agent(user_input)
    
    return response_agent(category, worker_output, user_input)

# ============================================
# DYNAMIC RUN LOOP
# ============================================
print(" Decisive Multi-Agent System Online (Resolution Mode)\n")

while True:
    user_query = input("Enter issue: ").strip()
    if user_query.lower() in ["exit", "quit"]: break
    if not user_query: continue

    print(run_agentic_workflow(user_query))

 Decisive Multi-Agent System Online (Resolution Mode)

 [Classifier] is processing...
 [Billing Agent] is processing...

--------------------------------------------------
 AGENTIC RESOLUTION SUMMARY
--------------------------------------------------
DEPT    : BILLING
INPUT   : I have refund issue of 500$ of one product where i paid through my card
--------------------------------------------------
ACTION TAKEN:
I acknowledge that the specific amount of $500 is related to the refund issue you are experiencing, and I will ensure that this amount is processed accordingly.

I confirm that a refund of $500 has been INITIATED in our system. 

No further action is needed from you, and you can expect the refund to be processed in accordance with our standard refund procedures.
--------------------------------------------------
STATUS  : CLOSED / RESOLVED
--------------------------------------------------

